In [1]:
import pandas as pd
import duckdb

# Read both sheets of the Excel file (2009-2010 and 2010-2011) and stack them
sheets = pd.read_excel("../data/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

# Clean column names: "Customer ID" -> "customer_id", etc.
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Invoice and StockCode mix numbers and text, so store them as text
df["invoice"] = df["invoice"].astype(str)
df["stockcode"] = df["stockcode"].astype(str)

# Save into a DuckDB database file so we never read the slow Excel again
con = duckdb.connect("../data/retail.duckdb")
con.execute("CREATE OR REPLACE TABLE raw_sales AS SELECT * FROM df")

print("Sheets:", list(sheets.keys()))
print("Rows:", len(df))
print(df.dtypes)

Sheets: ['Year 2009-2010', 'Year 2010-2011']
Rows: 1067371
invoice                   str
stockcode                 str
description            object
quantity                int64
invoicedate    datetime64[us]
price                 float64
customer_id           float64
country                   str
dtype: object


In [2]:
quality = con.sql("""
    SELECT
        COUNT(*)                                          AS total_rows,
        SUM(CASE WHEN invoice LIKE 'C%' THEN 1 ELSE 0 END) AS cancelled_rows,
        SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END)     AS zero_or_negative_qty,
        SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END)        AS zero_or_negative_price,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer,
        SUM(CASE WHEN description IS NULL THEN 1 ELSE 0 END) AS missing_description
    FROM raw_sales
""").df()

duplicates = con.sql("""
    SELECT COUNT(*) - COUNT(DISTINCT (invoice, stockcode, description, quantity, invoicedate, price, customer_id, country)) AS duplicate_rows
    FROM raw_sales
""").df()

print(quality.T)
print(duplicates)

                                0
total_rows              1067371.0
cancelled_rows            19494.0
zero_or_negative_qty      22950.0
zero_or_negative_price     6207.0
missing_customer         243007.0
missing_description        4382.0
   duplicate_rows
0           34335


In [7]:
con.execute("""
    CREATE OR REPLACE TABLE clean_sales AS
    SELECT DISTINCT invoice, stockcode, description, quantity, invoicedate, price,
           CAST(customer_id AS INTEGER) AS customer_id, country,
           quantity * price AS revenue
    FROM raw_sales
    WHERE invoice NOT LIKE 'C%'
      AND quantity > 0
      AND price > 0
      AND customer_id IS NOT NULL
      AND regexp_matches(stockcode, '^[0-9]{5}')
""")

In [8]:
summary = con.sql("""
    SELECT COUNT(*) AS rows_kept,
           COUNT(DISTINCT customer_id) AS customers,
           COUNT(DISTINCT invoice) AS orders,
           COUNT(DISTINCT stockcode) AS products,
           COUNT(DISTINCT country) AS countries,
           ROUND(SUM(revenue), 0) AS total_revenue_gbp,
           MIN(invoicedate) AS first_date,
           MAX(invoicedate) AS last_date
    FROM clean_sales
""").df()
print(summary.T)


                                     0
rows_kept                       776577
customers                         5852
orders                           36594
products                          4619
countries                           41
total_revenue_gbp           17068568.0
first_date         2009-12-01 07:45:00
last_date          2011-12-09 12:50:00
